# Embedding maps

t-SNE / PCA of the key spaces, before and after LIBERO fine-tuning.

In [ ]:
import json
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D

import utils as u

FS_SMALL, FS_TINY = 9, 7  # pt: \small, \scriptsize of the paper
PAGE_WIDTH = 5.5  # in: \textwidth of the paper (\linewidth of a figure)
ROOT = Path.cwd().parent
data_path = ROOT / "results"
figs_path = ROOT / "analysis/figs"
style_path = ROOT / "analysis/figure.mplstyle"

In [ ]:
maps = json.loads((data_path / "embedding_maps.json").read_text())
figs_path.mkdir(exist_ok=True)
plt.style.use(style_path)

In [ ]:
MAP_POINT_SIZE = 1.2  # embedding map scatter
SUITE_COLOURS = dict(zip(u.SUITES, mpl.color_sequences["petroff6"], strict=False))  # vivid, distinct for colour-blind readers


In [ ]:
def map_panel(ax: mpl.axes.Axes, space: str, method: str, colour_by: str, stats: bool = True) -> None:
    """One key space's 2-D map (method "tsne" or "pca"), points coloured by episode progress or by suite. t-SNE maps are
    labelled with the phase gap, and with ``stats`` the same-task neighbours overall and on LIBERO-Goal
    (embedding_maps.py); PCA maps with the variance share of the suite and of the two components."""
    xy = np.array(maps["spaces"][space][method])
    if colour_by == "progress":
        cmap = mpl.colormaps.get_cmap("plasma")
        c = cmap(np.array(maps["progress"]))
    else:
        c = np.array([mpl.colors.to_rgba(SUITE_COLOURS[s]) for s in maps["suite"]])
    order = np.random.default_rng(0).permutation(len(xy))  # no suite / phase drawn last on top of the others
    ax.scatter(*xy[order].T, s=MAP_POINT_SIZE, c=c[order], linewidths=0, rasterized=True)
    ax.set_xticks([])
    ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)
    ax.set_aspect("equal", adjustable="datalim")
    v = maps["spaces"][space]
    label = f"Gap {v['progress_gap']:.3f}"
    if stats:
        label = f"Task {100 * v['task_nn']:.0f}% · Goal {100 * v['task_nn/libero_goal']:.0f}%\n" + label
    if method == "pca":
        label = f"Suite {100 * u.suite_r2(maps, space):.0f}% · PC1+2 {100 * sum(v['pca_variance']):.0f}%"
    ax.set_xlabel(label, fontsize=FS_TINY)


def map_legend(fig, colour_by: str) -> None:
    if colour_by == "progress":
        cmap = mpl.colormaps["plasma"]
        sm = mpl.cm.ScalarMappable(cmap=cmap, norm=mpl.colors.Normalize(0, 1))
        cb = fig.colorbar(sm, ax=fig.axes, location="bottom", shrink=0.35, aspect=30, pad=0.02)
        cb.set_ticks([0, 1], labels=["Start", "End"])
        cb.ax.set_xlabel("Episode", labelpad=0)
        cb.ax.tick_params(labelsize=FS_TINY, length=0)
        cb.outline.set_visible(False)
    else:
        names = {
            "libero_spatial": "Spatial",
            "libero_object": "Object",
            "libero_goal": "Goal",
            "libero_10": "Long",
        }
        handles = [Line2D([], [], marker="o", linestyle="", color=c, markersize=6, label=names[s])
                   for s, c in SUITE_COLOURS.items()]  # fmt: skip
        fig.legend(handles=handles, frameon=False, loc="outside lower center", ncol=4, fontsize=FS_TINY)

In [ ]:
MAP_GRID = [  # (key space, panel title): generic keys and GR00T's frozen VLM, then each VLA before / after fine-tuning
    [("ResNet-18 grid", "ResNet-18"), ("ResNet-18 grid (learnt key)", "$k$-NN (LAC key)"), ("DINOv2-S", "DINOv2"),
     ("GR00T N1.7 frozen VLM grid2@L4", "GR00T (frozen)")],
    [("π0.5 base L12", r"$\pi_{0.5}$ before"), ("π0.5 LIBERO L12", r"$\pi_{0.5}$ after"),
     ("MolmoAct2 base grid2@L27", "MolmoAct2 before"), ("MolmoAct2 LIBERO grid2@L27", "MolmoAct2 after")],
    [("VLA-JEPA base grid2@L7", "VLA-JEPA before"), ("VLA-JEPA LIBERO grid2@L7", "VLA-JEPA after"),
     ("Evo-1 base grid2@L10", "Evo-1 before"), ("Evo-1 LIBERO grid2@L10", "Evo-1 after")],
    [("X-VLA base grid2@L8", "X-VLA before"), ("X-VLA LIBERO grid2@L8", "X-VLA after"),
     ("π0-FAST base grid2@L12", r"$\pi_0$-FAST before"), ("π0-FAST LIBERO grid2@L12", r"$\pi_0$-FAST after")],
    [("SmolVLA base grid", "SmolVLA before"), ("SmolVLA LIBERO grid", "SmolVLA after"),
     ("π0 base L12", r"$\pi_0$ before"), ("π0 LIBERO L12", r"$\pi_0$ after")],
]  # fmt: skip


MAP_PAIRS = [  # the compact figure: (VLA, key before fine-tuning, after)
    (r"$\pi_{0.5}$", "π0.5 base L12", "π0.5 LIBERO L12"),
    ("Evo-1", "Evo-1 base grid2@L10", "Evo-1 LIBERO grid2@L10"),
    ("MolmoAct2", "MolmoAct2 base grid2@L27", "MolmoAct2 LIBERO grid2@L27"),
    (r"$\pi_0$", "π0 base L12", "π0 LIBERO L12"),
]

## Before / after pairs

t-SNE of a few VLMs before (top) and after (bottom) LIBERO fine-tuning, coloured by episode progress, next to the ImageNet ResNet-18 (top) and the key learnt for LAC (bottom).

In [ ]:
cols = [("ResNet-18", "ResNet-18 grid", "ResNet-18 grid (learnt key)"), *MAP_PAIRS]
fig, axes = plt.subplots(2, len(cols), figsize=(PAGE_WIDTH, PAGE_WIDTH * 0.38), layout="constrained")
for j, (name, before, after) in enumerate(cols):
    for i, space in enumerate((before, after)):
        map_panel(axes[i, j], space, "tsne", "progress", stats=False)
    axes[0, j].set_title(name)
axes[0, 0].set_ylabel("Before")
axes[1, 0].set_ylabel("After")
fig.supylabel("Training")
axes[1, 0].set_title("$k$-NN (LAC key)", color=u.Colour.KNN)
map_legend(fig, "progress")
fig.savefig(figs_path / "embedding_pairs.pdf", dpi=300)  # the scatter layers are rasterised

## Appendix: every key

The generic keys and GR00T's frozen VLM, then each VLA's VLM before / after fine-tuning.

In [ ]:
method, colour_by = "tsne", "progress"
fig, axes = plt.subplots(len(MAP_GRID), 4, figsize=(PAGE_WIDTH, PAGE_WIDTH * 1.32), layout="constrained")
for row, panels in zip(axes, MAP_GRID, strict=True):
    for ax, (space, title) in zip(row, panels, strict=True):
        map_panel(ax, space, method, colour_by)
        ax.set_title(title)
map_legend(fig, colour_by)
fig.savefig(figs_path / f"embedding_{method}_{colour_by}.pdf", dpi=300)  # the scatter layers are rasterised

In [ ]:
method, colour_by = "pca", "suite"
fig, axes = plt.subplots(len(MAP_GRID), 4, figsize=(PAGE_WIDTH, PAGE_WIDTH * 1.32), layout="constrained")
for row, panels in zip(axes, MAP_GRID, strict=True):
    for ax, (space, title) in zip(row, panels, strict=True):
        map_panel(ax, space, method, colour_by)
        ax.set_title(title)
map_legend(fig, colour_by)
fig.savefig(figs_path / f"embedding_{method}_{colour_by}.pdf", dpi=300)  # the scatter layers are rasterised